# Visualisation II: Further Dimensionality Reduction

### [Neil D. Lawrence](http://inverseprobability.com), University of

Cambridge

### 2024-11-13

**Abstract**: Building on our understanding of discrete and continuous
latent variables, this lecture explores modern approaches to
dimensionality reduction. We examine the limitations of linear methods
like PCA, introduce powerful nonlinear techniques like t-SNE, and
develop practical guidelines for choosing and implementing these
methods. The lecture emphasizes the importance of understanding when
methods preserve local versus global structure and how this affects
their application.

$$
$$

<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!---->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!-- The last names to be defined. Should be defined entirely in terms of macros from above-->
<!--

-->

## Setup

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_notebooks/includes/notebook-setup.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_notebooks/includes/notebook-setup.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import matplotlib.pyplot as plt
import shutil

if shutil.which('latex') is None:
    plt.rcParams['text.usetex'] = False
else:
    plt.rcParams['text.usetex'] = True
    plt.rcParams['text.latex.preamble']=r'\usepackage{amsmath}'

plt.rcParams.update({'font.size': 22})

<!--setupplotcode{import seaborn as sns
sns.set_style('darkgrid')
sns.set_context('paper')
sns.set_palette('colorblind')}-->

## notutils

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_software/includes/notutils-software.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_software/includes/notutils-software.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

This small package is a helper package for various notebook utilities
used below.

The software can be installed using

In [ ]:
import importlib.util

In [ ]:
def install_command(imp):
    packages = {
        'daft': 'daft-pgm',
        'mlai': 'git+https://github.com/lawrennd/mlai.git',
        'pods': 'pods',
        'PyDeepGP': 'git+https://github.com/SheffieldML/PyDeepGP.git',
        'notutils': 'notutils',
        'pods' : 'git+https://github.com/lawrennd/ods.git',
        'lamd': 'lamd',
        'GPy': 'gpy',
    'emukit': 'emukit',
    }

    if importlib.util.find_spec(imp) is None:
        if imp in packages:
            return f'pip install {packages[imp]}'
        else:
            return f'pip install {imp}'
    else:
        return f'echo "Package {imp} is already installed."'

from the command prompt where you can access your python installation.

The code is also available on GitHub:
<https://github.com/lawrennd/notutils>

Once `notutils` is installed, it can be imported in the usual manner.

In [ ]:
import notutils

## pods

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_software/includes/pods-software.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_software/includes/pods-software.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In Sheffield we created a suite of software tools for ‘Open Data
Science.’ Open data science is an approach to sharing code, models and
data that should make it easier for companies, health professionals and
scientists to gain access to data science techniques.

You can also check this blog post on [Open Data
Science](http://inverseprobability.com/2014/07/01/open-data-science).

The software can be installed using

In [ ]:
cmd = install_command('pods')

In [ ]:
%system {cmd}

from the command prompt where you can access your python installation.

The code is also available on GitHub: <https://github.com/lawrennd/ods>

Once `pods` is installed, it can be imported in the usual manner.

In [ ]:
import pods

## mlai

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_software/includes/mlai-software.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_software/includes/mlai-software.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

The `mlai` software is a suite of helper functions for teaching and
demonstrating machine learning algorithms. It was first used in the
Machine Learning and Adaptive Intelligence course in Sheffield in 2013.

The software can be installed using

In [ ]:
cmd = install_command('mlai')

In [ ]:
%system {cmd}

from the command prompt where you can access your python installation.

The code is also available on GitHub: <https://github.com/lawrennd/mlai>

Once `mlai` is installed, it can be imported in the usual manner.

In [ ]:
import mlai
from mlai import plot

# Part 1: Linear PCA

## Principal Component Analysis

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/principal-component-analysis.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/principal-component-analysis.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Principal Component Analysis (PCA) is one of the most fundamental and
widely used dimensionality reduction techniques. While commonly credited
to Pearson (1901), who was interested in finding “lines and planes of
closest fit to systems of points in space,” the method as we will review
it today was introduced and *named* by Hotelling (Hotelling (1933)). The
approach is often used as a way of looking for correlations, but
Hotelling’s motivation was a principled alternative to Spearman’s factor
analysis (Spearman, 1904).

Often PCA is introduced as a method for finding directions of maximum
variance in high-dimensional data, and this is the interpretation that
is due to Pearson (1901), but philosophically these approaches arre
different even though they turn out to be identical mathematically. In a
very real sense “many models lead to the PCA algorithm.” But these
equivalences are only true when a *linear* interpretation is sought.
Nonlinear extensions of these ideas (maximum variance directions,
eigenvalue problems, latent variable models) all lead to *different*
algorithms. However, since the linear algorithm has so many
interpretations it is a wise place to begin analysis.

The mathematical foundation of PCA relies on analyzing the sample
covariance matrix of the data. For a dataset with $n$ points, this
matrix is given by:

$$
\mathbf{S}=\frac{1}{n}\sum_{i=1}^n\left(\mathbf{ y}_{i, :}-\boldsymbol{ \mu}\right)\left(\mathbf{ y}_{i, :} - \boldsymbol{ \mu}\right)^\top
$$

The modern interpretation focuses on finding a set of orthogonal
directions (principal components) along which the data varies the most,
but Hotelling’s original formulation was derived from the idea of latent
variables. The optimization problem we solve today, which maximizes the
variance along each direction while maintaining orthogonality
constraints, arises as the solution of the original latent variable
formulation (Tipping and Bishop, 1999a).

In [ ]:
import numpy as np

In [ ]:
# Generate correlated 2D Gaussian samples
num_points = 200
mean = np.array([0, 0])
cov = np.array([[2.0, 1.8], 
                [1.8, 2.0]])
X = np.random.multivariate_normal(mean, cov, num_points)

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot

In [ ]:
def pca_plot(ax, X):
    # Compute eigenvalues and eigenvectors of covariance matrix
    eigvals, eigvecs = np.linalg.eigh(np.cov(X.T))
    
    # Plot data points
    ax.scatter(X[:, 0], X[:, 1], alpha=0.5)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.axis('equal')

    mlai.write_figure("pca-directions-000.svg", directory='./dimred')

    # Plot covariance ellipse
    # Get eigenvalues and eigenvectors of covariance matrix
    eigenvals, eigenvecs = np.linalg.eigh(cov)
    
    # Calculate angle of rotation from largest eigenvector
    angle = np.arctan2(eigenvecs[1,1], eigenvecs[0,1])
    
    # Calculate width and height of ellipse from eigenvalues
    width = 2 * np.sqrt(eigenvals[1])  # 2 std deviations
    height = 2 * np.sqrt(eigenvals[0])
    
    # Create and add ellipse patch
    ellipse = plt.matplotlib.patches.Ellipse(mean, width, height,
                                           angle=angle * 180/np.pi,
                                           facecolor='none',
                                           edgecolor='r',
                                           linestyle='--')
    ax.add_patch(ellipse)
    mlai.write_figure("pca-directions-001.svg", directory='./dimred')
    
    # Plot principal components
    counter = 1
    for i in [0, 1]:
        counter += 1
        ax.arrow(mean[0], mean[1], 
                eigvecs[0, i]*np.sqrt(eigvals[i]), 
                eigvecs[1, i]*np.sqrt(eigvals[i]),
                head_width=0.1, head_length=0.1, fc='k', ec='k')
        mlai.write_figure(f"pca-directions-{counter:03d}.svg", directory='./dimred')

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
pca_plot(ax, X)

In [ ]:
import notutils as nu

In [ ]:
nu.display_plots("pca-directions-{counter:0>3}.svg", directory="./dimred", counter=(0, 3))

<img src="https://mlatcl.github.io/advds/diagrams/dimred/pca-directions-003.svg" class="" width="\width" style="vertical-align:middle;">

Figure: <i>Illustration of PCA on 2D correlated Gaussian data. The
arrows show the principal components (eigenvectors scaled by square root
of eigenvalues). The ellipse represents the covariance structure.</i>

## Probabilistic PCA

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/probabilistic-pca.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/probabilistic-pca.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

This linear relationship between the observed data and the latent
variables is at the heart of Hotelling’s original formulation of PCA. It
explicitly models the idea that the observed high-dimensional data is
generated from a lower-dimensional set of latent variables, with some
added noise. This perspective aligns PCA more closely with factor
analysis and highlights its nature as a latent variable model.

Probabilistic PCA (PPCA) provides a probabilistic interpretation of PCA
by explicitly modeling the generative process that creates the observed
data. The model assumes that each high-dimensional data point
$\mathbf{ y}_{i,:}$ is generated from a lower-dimensional latent
variable $\mathbf{ z}_{i,:}$ through a linear transformation with added
Gaussian noise: $$
\mathbf{ y}_{i,:} = \mathbf{W}\mathbf{ z}_{i,:} + \boldsymbol{ \epsilon}_{i,:}
$$ where
$\boldsymbol{ \epsilon}_{i,:} \sim \mathscr{N}\left(\mathbf{0},\sigma^2\mathbf{I}\right)$
represents isotropic Gaussian noise. The model places a standard
Gaussian prior over the latent variables: $$
p(\mathbf{ z}_{i,:}) = \mathscr{N}\left(\mathbf{ z}_{i,:}|\mathbf{0},\mathbf{I}\right)
$$ Given these assumptions, the conditional probability of observing a
data point given its latent representation is: $$
p(\mathbf{ y}_{i,:}|\mathbf{ z}_{i,:},\mathbf{W}) = \mathscr{N}\left(\mathbf{ y}_{i,:}|\mathbf{W}\mathbf{ z}_{i,:},\sigma^2\mathbf{I}\right)
$$ By integrating out the latent variables, we obtain the marginal
likelihood of the data: $$
p(\mathbf{ y}_{i,:}|\mathbf{W}) = \mathscr{N}\left(\mathbf{ y}_{i,:}|\mathbf{0},\mathbf{W}\mathbf{W}^{\top}+\sigma^{2}\mathbf{I}\right)
$$ This probabilistic formulation, developed by Tipping and Bishop
(Tipping and Bishop, 1999a), provides a principled framework that not
only recovers classical PCA as a special case when $\sigma^2 \to 0$, but
also enables extensions like handling missing data and mixture models.

In [ ]:
from mlai import plot
import mlai
from matplotlib import pyplot as plt

In [ ]:
pgm = plot.ppca_graphical_model()
filename = mlai.filename_join('ppca_graphical_model.svg', directory='./dimred')
pgm.render().figure.savefig(filename, transparent=True)

<img src="https://mlatcl.github.io/advds/diagrams/dimred/ppca_graphical_model.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>Graphical model representing probabilistic PCA.</i>

## Probabilistic PCA

In 1997 [Tipping and
Bishop](http://research.microsoft.com/pubs/67218/bishop-ppca-jrss.pdf)
(Tipping and Bishop, 1999b) and
[Roweis](https://www.cs.nyu.edu/~roweis/papers/empca.pdf) (Roweis, n.d.)
independently revisited Hotelling’s model and considered the case where
the noise variance was finite, but *shared* across all output dimensons.
Their model can be thought of as a factor analysis where $$
\boldsymbol{\Sigma} = \sigma^2 \mathbf{I}.
$$ This leads to a marginal likelihood of the form $$
p(\mathbf{Y}|\mathbf{W}, \sigma^2)
= \prod_{i=1}^n\mathscr{N}\left(\mathbf{ y}_{i, :}|\mathbf{0},\mathbf{W}\mathbf{W}^\top + \sigma^2 \mathbf{I}\right)
$$ where the limit of $\sigma^2\rightarrow 0$ is *not* taken. This
defines a proper probabilistic model. Tippping and Bishop then went on
to prove that the *maximum likelihood* solution of this model with
respect to $\mathbf{W}$ is given by an eigenvalue problem. In the
probabilistic PCA case the eigenvalues and eigenvectors are given as
follows. $$
\mathbf{W}= \mathbf{U}\mathbf{L} \mathbf{R}^\top
$$ where $\mathbf{U}$ is the eigenvectors of the empirical covariance
matrix $$
\mathbf{S}= \sum_{i=1}^n(\mathbf{ y}_{i, :} - \boldsymbol{ \mu})(\mathbf{ y}_{i,:} - \boldsymbol{ \mu})^\top,
$$ which can be written
$\mathbf{S}= \frac{1}{n} \mathbf{Y}^\top\mathbf{Y}$ if the data is zero
mean. The matrix $\mathbf{L}$ is diagonal and is dependent on the
*eigenvalues* of $\mathbf{S}$, $\boldsymbol{\Lambda}$. If the $i$th
diagonal element of this matrix is given by $\lambda_i$ then the
corresponding element of $\mathbf{L}$ is $$
\ell_i = \sqrt{\lambda_i - \sigma^2}
$$ where $\sigma^2$ is the noise variance. Note that if $\sigma^2$ is
larger than any particular eigenvalue, then that eigenvalue (along with
its corresponding eigenvector) is *discarded* from the solution.

## PPCA as Manifold Learning

Probabilistic PCA can be viewed as a simple manifold learning algorithm.
It assumes that:

1.  The data lies near a linear manifold (subspace) of the
    high-dimensional space
2.  The deviation from this manifold is Gaussian noise
3.  The intrinsic dimensionality is specified by the number of retained
    components

This view helps explain why PPCA works well when the manifold hypothesis
holds and the manifold is approximately linear. When the manifold is
nonlinear, we need more sophisticated methods like kernel PCA or neural
network-based approaches.

## Python Implementation of Probabilistic PCA

We will now implement this algorithm in python.

In [ ]:
import numpy as np

In [ ]:
from mlai import ppca_eig
import inspect
file_path = inspect.getfile(ppca_eig)

In [ ]:
%load -s ppca_eig {file_path}

In [ ]:
ppca = ppca_eig

In practice we may not wish to compute the eigenvectors of the
covariance matrix directly. This is because it requires us to estimate
the covariance, which involves a sum of squares term, before estimating
the eigenvectors. We can estimate the eigenvectors directly either
through [QR
decomposition](http://en.wikipedia.org/wiki/QR_decomposition) or
[singular value
decomposition](http://en.wikipedia.org/wiki/Singular_value_decomposition).
We saw a similar issue arise when , where we also wished to avoid
computation of $\mathbf{Z}^\top\mathbf{Z}$ (or in the case of
$\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$).

## Posterior for Principal Component Analysis

Under the latent variable model justification for principal component
analysis, we are normally interested in inferring something about the
latent variables given the data. This is the distribution, $$
p(\mathbf{ z}_{i, :} | \mathbf{ y}_{i, :})
$$ for any given data point. Determining this density turns out to be
very similar to the approach for determining the Bayesian posterior of
$\mathbf{ w}$ in Bayesian linear regression, only this time we place the
prior density over $\mathbf{ z}_{i, :}$ instead of $\mathbf{ w}$. The
posterior is proportional to the joint density as follows, $$
p(\mathbf{ z}_{i, :} | \mathbf{ y}_{i, :}) \propto p(\mathbf{ y}_{i,
:}|\mathbf{W}, \mathbf{ z}_{i, :}, \sigma^2) p(\mathbf{ z}_{i, :})
$$ And as in the Bayesian linear regression case we first consider the
log posterior, $$
\log p(\mathbf{ z}_{i, :} | \mathbf{ y}_{i, :}) = \log p(\mathbf{ y}_{i, :}|\mathbf{W},
\mathbf{ z}_{i, :}, \sigma^2) + \log p(\mathbf{ z}_{i, :}) + \text{const}
$$ where the constant is not dependent on $\mathbf{ z}$. As before we
collect the quadratic terms in $\mathbf{ z}_{i, :}$ and we assemble them
into a Gaussian density over $\mathbf{ z}$. $$
\log p(\mathbf{ z}_{i, :} | \mathbf{ y}_{i, :}) =
-\frac{1}{2\sigma^2} (\mathbf{ y}_{i, :} - \mathbf{W}\mathbf{ z}_{i,
:})^\top(\mathbf{ y}_{i, :} - \mathbf{W}\mathbf{ z}_{i, :}) - \frac{1}{2}
\mathbf{ z}_{i, :}^\top \mathbf{ z}_{i, :} + \text{const}
$$

### Exercise 1

Multiply out the terms in the brackets. Then collect the quadratic term
and the linear terms together. Show that the posterior has the form $$
\mathbf{ z}_{i, :} | \mathbf{W}\sim \mathscr{N}\left(\boldsymbol{ \mu}_x,\mathbf{C}_x\right)
$$ where $$
\mathbf{C}_x = \left(\sigma^{-2}
\mathbf{W}^\top\mathbf{W}+ \mathbf{I}\right)^{-1}
$$ and $$
\boldsymbol{ \mu}_x
= \mathbf{C}_x \sigma^{-2}\mathbf{W}^\top \mathbf{ y}_{i, :} 
$$ Compare this to the posterior for the Bayesian linear regression from
last week, do they have similar forms? What matches and what differs?

### Exercise 1 Answer

Write your answer to Exercise 1 here

## Python Implementation of the Posterior

Now let’s implement the system in code.

### Exercise 2

Use the values for $\mathbf{W}$ and $\sigma^2$ you have computed, along
with the data set $\mathbf{Y}$ to compute the posterior density over
$\mathbf{Z}$. Write a function of the form

``` python
mu_x, C_x = posterior(Y, W, sigma2)
```

where `mu_x` and `C_x` are the posterior mean and posterior covariance
for the given $\mathbf{Y}$.

Don’t forget to subtract the mean of the data `Y` inside your function
before computing the posterior: remember we assumed at the beginning of
our analysis that the data had been centred (i.e. the mean was removed).

In [ ]:
# Write your answer to Exercise 2 here


# Answer Code
# Write code for you answer to this exercise in this box
# Do not delete these comments, otherwise you will get zero for this answer.
# Make sure your code has run and the answer is correct *before* submitting your notebook for marking.
import numpy as np
import scipy as sp
def posterior(Y, W, sigma2):
    Y_cent = Y - Y.mean(0)
    # Compute posterior over X
    C_x = 
    mu_x = 
    return mu_x, C_x



## Numerically Stable and Efficient Version

Just as we saw for and computation of a matrix such as
$\mathbf{Y}^\top\mathbf{Y}$ (or its centred version) can be a bad idea
in terms of loss of numerical accuracy. Fortunately, we can find the
eigenvalues and eigenvectors of the matrix $\mathbf{Y}^\top\mathbf{Y}$
without direct computation of the matrix. This can be done with the
[*singular value
decomposition*](http://en.wikipedia.org/wiki/Singular_value_decomposition).
The singular value decompsition takes a matrix, $\mathbf{Z}$ and
represents it in the form, $$
\mathbf{Z} = \mathbf{U}\boldsymbol{\Lambda}\mathbf{V}^\top
$$ where $\mathbf{U}$ is a matrix of orthogonal vectors in the columns,
meaning $\mathbf{U}^\top\mathbf{U} = \mathbf{I}$. It has the same number
of rows and columns as $\mathbf{Z}$. The matrices $\mathbf{\Lambda}$ and
$\mathbf{V}$ are both square with dimensionality given by the number of
columns of $\mathbf{Z}$. The matrix $\mathbf{\Lambda}$ is *diagonal* and
$\mathbf{V}$ is an orthogonal matrix so
$\mathbf{V}^\top\mathbf{V} = \mathbf{V}\mathbf{V}^\top = \mathbf{I}$.
The eigenvalues of the matrix $\mathbf{Y}^\top\mathbf{Y}$ are then given
by the singular values of the matrix $\mathbf{Y}^\top$ squared and the
eigenvectors are given by $\mathbf{U}$.

## Solution for $\mathbf{W}$

Given the singular value decomposition of $\mathbf{Y}$ then we have $$
\mathbf{W}=
\mathbf{U}\mathbf{L}\mathbf{R}^\top
$$ where $\mathbf{R}$ is an arbitrary rotation matrix. This implies that
the posterior is given by $$
\mathbf{C}_x =
\left[\sigma^{-2}\mathbf{R}\mathbf{L}^2\mathbf{R}^\top + \mathbf{I}\right]^{-1}
$$ because $\mathbf{U}^\top \mathbf{U} = \mathbf{I}$. Since, by
convention, we normally take $\mathbf{R} = \mathbf{I}$ to ensure that
the principal components are orthonormal we can write $$
\mathbf{C}_x = \left[\sigma^{-2}\mathbf{L}^2 +
\mathbf{I}\right]^{-1}
$$ which implies that $\mathbf{C}_x$ is actually diagonal with elements
given by $$
c_i = \frac{\sigma^2}{\sigma^2 + \ell^2_i}
$$ and allows us to write $$
\boldsymbol{ \mu}_x = [\mathbf{L}^2 + \sigma^2
\mathbf{I}]^{-1} \mathbf{L} \mathbf{U}^\top \mathbf{ y}_{i, :}
$$ $$
\boldsymbol{ \mu}_x = \mathbf{D}\mathbf{U}^\top \mathbf{ y}_{i, :}
$$ where $\mathbf{D}$ is a diagonal matrix with diagonal elements given
by $d_{i} = \frac{\ell_i}{\sigma^2 + \ell_i^2}$.

In [ ]:
import scipy as sp
import numpy as np

In [ ]:
from mlai import ppca_svd
import inspect
file_path = inspect.getfile(ppca_svd)

In [ ]:
%load -s ppca_svd {file_path}

In [ ]:
from mlai import ppca_posterior
import inspect
file_path = inspect.getfile(ppca_posterior)

In [ ]:
%load -s ppca_posterior {file_path}

In [ ]:
ppca = ppca_svd
posterior = ppca_posterior

## Scikit-learn implementation PCA

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/oil-flow-sklearn-pca.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/oil-flow-sklearn-pca.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We’ve implemented PCA as part of supporting the learning process, but in
practice we can use the `scikit-learn` implementation. Let’s try it on
the oil flow data.

## Oil Flow Data

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_datasets/includes/oil-flow-data.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_datasets/includes/oil-flow-data.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

This data set is from a physics-based simulation of oil flow in a
pipeline. The data was generated as part of a project to determine the
fraction of oil, water and gas in North Sea oil pipes (Bishop and James,
1993).

In [ ]:
import pods

In [ ]:
data = pods.datasets.oil()

The data consists of 1000 12-dimensional observations of simulated oil
flow in a pipeline. Each observation is labelled according to the
multi-phase flow configuration (homogeneous, annular or laminar).

In [ ]:
# Convert data["Y"] from [1, -1, -1] in each row to rows of 0 or 1 or 2
Y = data["Y"]
# Find rows with 1 in first column (class 0)
class0 = (Y[:, 0] == 1).astype(int) * 0
# Find rows with 1 in second column (class 1) 
class1 = (Y[:, 1] == 1).astype(int) * 1
# Find rows with 1 in third column (class 2)
class2 = (Y[:, 2] == 1).astype(int) * 2
# Combine into single array of class labels 0,1,2
labels = class0 + class1 + class2

The data is returned as a dictionary containing training and test inputs
(‘X,’ ‘Xtst’), training and test labels (‘Y,’ ‘Ytst’), and the names of
the features.

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai
import numpy as np

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
# Plot first two dimensions of the data
classes = np.unique(labels)
colors = ['r', 'g', 'b']
labels = ["homogeneous", "annular", "stratified"]

for i, cls in enumerate(classes):
    idx = labels == cls
    ax.plot(data['X'][idx, 0], data['X'][idx, 1], colors[i] + '.', 
            markersize=10, label=f'{labels[i]}')
ax.set_xlabel('1st dimension')
ax.set_ylabel('2nd dimension')
ax.legend()

mlai.write_figure('oil-flow-data.svg', directory='./datasets')

<img src="https://mlatcl.github.io/advds/diagrams/datasets/oil-flow-data.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Visualization of the first two dimensions of the oil flow
data from Bishop and James (1993)</i>

As normal we include the citation information for the data.

In [ ]:
print(data['citation'])

And extra information about the data is included, as standard, under the
keys `info` and `details`.

In [ ]:
print(data['details'])

In [ ]:
X = data['X']
Y = data['Y']

In [ ]:
cmd = install_command('scikit-learn')

In [ ]:
%system {cmd}

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=2)
pca.fit(X)
X_pca = pca.transform(X)

In [ ]:
# Find rows with 1 in first column (class 0)
class0 = (Y[:, 0] == 1).astype(int) * 0
# Find rows with 1 in second column (class 1) 
class1 = (Y[:, 1] == 1).astype(int) * 1
# Find rows with 1 in third column (class 2)
class2 = (Y[:, 2] == 1).astype(int) * 2
# Combine into single array of class labels 0,1,2
lbls = class0 + class1 + class2

In [ ]:
from matplotlib import pyplot as plt
import mlai
from mlai import plot

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
# Three labels stored in Y
labels = ["homogeneous", "annular", "stratified"]
for i in range(3):
    ax.scatter(X_pca[lbls==i, 0], X_pca[lbls==i, 1], label=f'{labels[i]}')
ax.set_xlabel('First principal component')
ax.set_ylabel('Second principal component')

mlai.write_figure("oil-flow-pca-sklearn.svg", directory='./dimred')

<img src="https://mlatcl.github.io/advds/diagrams/dimred/oil-flow-pca_sklearn.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>PCA of the oil flow data. The flow is homogeneous, annular
and stratified.</i>

## Examples: Motion Capture Data

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/osu-run1-ppca.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/osu-run1-ppca.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

## OSU Motion Capture Data: Run 1

Motion capture data the Open Motion Data Project by The Ohio State
University Advanced Computing Center for the Arts and Design.
Historically the data website was found here
<http://accad.osu.edu/research/mocap/mocap_data.htm>, although it is now
missing. The centre website is here: <https://accad.osu.edu>.

In [ ]:
import pods

You can download different data from the site, here we download the
‘run1’ motion.

In [ ]:
data = pods.datasets.osu_run1()

The data dictionary contains the keys ‘Y’ and ‘connect,’ which represent
the data and connections that can be used to create the skeleton.

In [ ]:
data['Y'].shape

The data has often been used in talks demonstrating GP-LVM models and
comparing variants such as back constrained and temporal models.

In [ ]:
print(data['citation'])

And extra information about the data is included, as standard, under the
keys under `details`.

In [ ]:
print(data['details'])

In [ ]:
Y = data['Y']

Once the data is loaded in we can examine the first two principal
components as follows,

In [ ]:
q = 2
U, ell, sigma2 = ppca(Y, q)
mu_x, C_x = posterior(Y, U, ell, sigma2)

In [ ]:
import matplotlib.pyplot as plt
import mlai

In [ ]:
plt.plot(mu_x[:, 0], mu_x[:, 1], 'rx-')
mlai.write_figure("osu-run1-pca.svg", directory="./dimred/")

<img src="https://mlatcl.github.io/advds/diagrams/dimred/osu-run1-pca.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>First two principal components of motion capture data of an
individual running.</i>

Here because the data is a time course, we have connected points that
are neighbouring in time. This highlights the form of the run, which
involves 3 paces. This projects in our low dimensional space to 3 loops.
We can examin how much residual variance there is in the system by
looking at `sigma2`.

In [ ]:
print(sigma2)

# Part 2: Beyond Linear Methods

## When Dimensionality Reduction Fails

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/dimensionality-reduction-failure-modes.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/dimensionality-reduction-failure-modes.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

While dimensionality reduction is powerful, it’s important to understand
when it can fail:

1.  When the data really is high dimensional with no simpler structure
2.  When the relationship between dimensions is highly nonlinear
3.  When different parts of the data have different intrinsic
    dimensionality

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

## The Swiss Roll Example

In [ ]:
# Generate data that lies on a Swiss roll manifold
def generate_swiss_roll(n_points=1000):
    t = 1.5 * np.pi * (1 + 2 * np.random.rand(n_points))
    y = 21 * np.random.rand(n_points)
    x = t * np.cos(t)
    z = t * np.sin(t)
    return np.column_stack((x, y, z)), t

X, t = generate_swiss_roll()

fig = plt.figure(figsize=plot.big_wide_figsize)
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=t, cmap='viridis')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$') 
ax.set_zlabel('$z$')
plt.colorbar(scatter)

mlai.write_figure('swiss-roll.svg', directory='./dimred')

<img src="https://mlatcl.github.io/advds/diagrams/dimred/swiss-roll.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>The Swiss roll dataset is a classic example of data with
nonlinear structure. The color represents the position along the roll,
showing how points that are far apart in the original space are actually
close in the intrinsic manifold.</i>

## Common Failure Modes

This example shows data lying on a Swiss roll manifold. Linear
dimensionality reduction methods like PCA will fail to capture the
structure of this data, while nonlinear methods like t-SNE or UMAP may
perform better.

Common failure modes include: 1. Using linear methods on nonlinear
manifolds 2. Assuming global structure when only local structure exists
3. Not accounting for noise in the data

<!--include{_dimred/includes/local-vs-global-preservation.md}-->
<!--include{_dimred/includes/iterative-dimensionality-reduction.md}-->

# Part 3: Graph Based Methods

## t-SNE

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/t-sne-intro.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/t-sne-intro.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

t-Distributed Stochastic Neighbor Embedding (t-SNE) is a dimensionality
reduction technique that focuses on preserving local structure. It
converts distances between points into probabilities and tries to match
these probabilities in the low-dimensional space.

The idea in t-SNE is to convert distances in data space into
probabilities. This is done through two stages. First a neighbourhood
graph is formed, so each point, $i$, is related to its $k$ nearest
neighbours.

If the squared distances between two data points are given by
$d_{i,j}^2$, then the probability of point $i$ linking to point $j$ is
given by, $$
p(r_{i,j} | \mathbf{ D}) = \pi_{i,j} =  \frac{\exp -\frac{d_{i,j}^2}{2\sigma^2}}{\sum_{k\in \mathscr{N}\left( i \right)}\exp -d_{i,j}}
$$ where $r_{i,j} = 1$ if the two points are linked, $\pi_{i,j}$ is the
probability of that link and the neighbourhood of $i$ is given by
$\mathscr{N}(i)$.

This is compared with probabilities in *latent* space which are given by
$$
p(s_{i,j} | \boldsymbol{ \Delta}) = q_{i,j} = (1 + \delta_{i,j})^(-1),
$$ where $s_{i,j} = 1$ if two points are linked in the latent space and
$q_{i,j}$ is the probability of that link. The latent distances are
given by $$
\delta_{i,j} = (\mathbf{ z}_{i, :} - \mathbf{ z}_{j, :})^\top*\mathbf{ z}_{i, :} - \mathbf{ z}_{j, :}).
$$}

$t$-SNE then minimises the KL divergence between these two link
distributions. $$
E(\mathbf{Z}= \sum_i \sum_{j\in\mathscr{N}\left( i \right)} p_{i,k} \log \frac{p_{i,k}}{q_{i,k}}
$$}

The approach was originally proposed by Hinton and Roweis (n.d.), then
later modified using the $t$-distribution in the latent space by van der
Maaten and Hinton (2008).

The basic idea is to minimise the Kullback-Leibler divergence between a
distribution defined in the latent space and a distribution defined in
the data space. The distribution is over neighbours in each space, by
definition the probability of point $j$ being selected as a neighbor of
point $i$ is:

$$p_{j|i} = \frac{\exp(-\|\mathbf{ y}_i - \mathbf{ y}_j\|^2/2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|\mathbf{ y}_i - \mathbf{ y}_k\|^2/2\sigma_i^2)}$$

where $\sigma_i$ is chosen to achieve a desired perplexity. In the
low-dimensional space, a t-distribution is used instead:

$$q_{ij} = \frac{(1 + \|\mathbf{ z}_i - \mathbf{ z}_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|\mathbf{ z}_k - \mathbf{ z}_l\|^2)^{-1}}$$

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
from mlai.plot import tsne_example
import inspect
file_path = inspect.getfile(tsne_example)

In [ ]:
%load -s tsne_example {file_path}

## Oil Flow Data

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_datasets/includes/oil-flow-data.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_datasets/includes/oil-flow-data.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

This data set is from a physics-based simulation of oil flow in a
pipeline. The data was generated as part of a project to determine the
fraction of oil, water and gas in North Sea oil pipes (Bishop and James,
1993).

In [ ]:
import pods

In [ ]:
data = pods.datasets.oil()

The data consists of 1000 12-dimensional observations of simulated oil
flow in a pipeline. Each observation is labelled according to the
multi-phase flow configuration (homogeneous, annular or laminar).

In [ ]:
# Convert data["Y"] from [1, -1, -1] in each row to rows of 0 or 1 or 2
Y = data["Y"]
# Find rows with 1 in first column (class 0)
class0 = (Y[:, 0] == 1).astype(int) * 0
# Find rows with 1 in second column (class 1) 
class1 = (Y[:, 1] == 1).astype(int) * 1
# Find rows with 1 in third column (class 2)
class2 = (Y[:, 2] == 1).astype(int) * 2
# Combine into single array of class labels 0,1,2
labels = class0 + class1 + class2

The data is returned as a dictionary containing training and test inputs
(‘X,’ ‘Xtst’), training and test labels (‘Y,’ ‘Ytst’), and the names of
the features.

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai
import numpy as np

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
# Plot first two dimensions of the data
classes = np.unique(labels)
colors = ['r', 'g', 'b']
labels = ["homogeneous", "annular", "stratified"]

for i, cls in enumerate(classes):
    idx = labels == cls
    ax.plot(data['X'][idx, 0], data['X'][idx, 1], colors[i] + '.', 
            markersize=10, label=f'{labels[i]}')
ax.set_xlabel('1st dimension')
ax.set_ylabel('2nd dimension')
ax.legend()

mlai.write_figure('oil-flow-data.svg', directory='./datasets')

<img src="https://mlatcl.github.io/advds/diagrams/datasets/oil-flow-data.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Visualization of the first two dimensions of the oil flow
data from Bishop and James (1993)</i>

As normal we include the citation information for the data.

In [ ]:
print(data['citation'])

And extra information about the data is included, as standard, under the
keys `info` and `details`.

In [ ]:
print(data['details'])

In [ ]:
X = data['X']
Y = data['Y']

In [ ]:
import matplotlib.pyplot as plt
import mlai
from mlai import plot

In [ ]:
tsne_example(X, Y)
mlai.write_figure('t-sne-oil-flow.svg', directory='./dimred')

<img src="https://mlatcl.github.io/advds/diagrams/dimred/t-sne-oil-flow.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>t-SNE embedding of the oil flow data.</i>

The perplexity parameter in t-SNE effectively controls how to balance
attention between local and global aspects of the data. It can be
interpreted as a smooth measure of the effective number of neighbors.

<!--include{_dimred/includes/umap-intro.md}-->

# Part 4: Practical Implementation

<!--include{_dimred/includes/dimensionality-reduction-comparison.md}-->

## Practical Tips for Dimensionality Reduction

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/dimensionality-reduction-practical-tips.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/dimensionality-reduction-practical-tips.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

While the theoretical foundations of dimensionality reduction are
important, success in practice often comes down to following some key
principles and understanding common pitfalls.

## Always Start with PCA

One of the most important practical tips for dimensionality reduction is
to always start with PCA:

There are several reasons for this:

1.  It’s fast and deterministic (up to reflections)
2.  It provides a baseline for more complex methods
3.  It can reveal linear structure you might have missed
4.  The eigenvalue spectrum gives you information about dimensionality
5.  It helps identify potential issues with your data

## Understanding Matrix Structure

Learn to extract information from key matrices:

Key patterns to look for:

1.  In Gram matrices:
    -   Block structure suggests clusters
    -   Diagonal dominance suggests noise
    -   Banded structure suggests ordering
2.  In distance matrices:
    -   Look for natural groupings
    -   Check for anomalous distances
    -   Verify triangle inequality

## Understanding Matrix Structure

Learn to extract information from key matrices. Here’s an example using
the oil flow data:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import mlai

In [ ]:
import pods

## Oil Flow Data

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_datasets/includes/oil-flow-data.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_datasets/includes/oil-flow-data.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

This data set is from a physics-based simulation of oil flow in a
pipeline. The data was generated as part of a project to determine the
fraction of oil, water and gas in North Sea oil pipes (Bishop and James,
1993).

In [ ]:
import pods

In [ ]:
data = pods.datasets.oil()

The data consists of 1000 12-dimensional observations of simulated oil
flow in a pipeline. Each observation is labelled according to the
multi-phase flow configuration (homogeneous, annular or laminar).

In [ ]:
# Convert data["Y"] from [1, -1, -1] in each row to rows of 0 or 1 or 2
Y = data["Y"]
# Find rows with 1 in first column (class 0)
class0 = (Y[:, 0] == 1).astype(int) * 0
# Find rows with 1 in second column (class 1) 
class1 = (Y[:, 1] == 1).astype(int) * 1
# Find rows with 1 in third column (class 2)
class2 = (Y[:, 2] == 1).astype(int) * 2
# Combine into single array of class labels 0,1,2
labels = class0 + class1 + class2

The data is returned as a dictionary containing training and test inputs
(‘X,’ ‘Xtst’), training and test labels (‘Y,’ ‘Ytst’), and the names of
the features.

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai
import numpy as np

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
# Plot first two dimensions of the data
classes = np.unique(labels)
colors = ['r', 'g', 'b']
labels = ["homogeneous", "annular", "stratified"]

for i, cls in enumerate(classes):
    idx = labels == cls
    ax.plot(data['X'][idx, 0], data['X'][idx, 1], colors[i] + '.', 
            markersize=10, label=f'{labels[i]}')
ax.set_xlabel('1st dimension')
ax.set_ylabel('2nd dimension')
ax.legend()

mlai.write_figure('oil-flow-data.svg', directory='./datasets')

<img src="https://mlatcl.github.io/advds/diagrams/datasets/oil-flow-data.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Visualization of the first two dimensions of the oil flow
data from Bishop and James (1993)</i>

As normal we include the citation information for the data.

In [ ]:
print(data['citation'])

And extra information about the data is included, as standard, under the
keys `info` and `details`.

In [ ]:
print(data['details'])

In [ ]:
X = data['X']
Y = data['Y']

In [ ]:
# Compute the Gram matrix
gram = X @ X.T

# Create a figure with two subplots
fig = plt.figure(figsize=plot.big_wide_figsize)
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1])

# Plot the Gram matrix
ax0 = plt.subplot(gs[0])
im = ax0.imshow(gram, cmap='viridis')
ax0.set_title('Gram Matrix')
plt.colorbar(im)

# Plot the first two dimensions of data
ax1 = plt.subplot(gs[1])
ax1.scatter(X[:, 0], X[:, 1], alpha=0.5)
ax1.set_title('Data Scatter Plot')
ax1.set_xlabel('Dimension 1')
ax1.set_ylabel('Dimension 2')

plt.tight_layout()
mlai.write_figure('matrix-structure.svg', directory='./dimred')

<img src="https://mlatcl.github.io/advds/diagrams/dimred/matrix-structure.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>The Gram matrix (left) reveals the underlying structure in
our data, while the scatter plot (right) shows the first two dimensions.
Notice how patterns in the Gram matrix correspond to clusters in the
data.</i>

The Gram matrix (left) reveals the underlying structure in our data,
while the scatter plot (right) shows the first two dimensions. Notice
how patterns in the Gram matrix correspond to clusters in the data.

## Examining Projections

When working with high-dimensional data, systematically examine
different projections:

<img src="https://mlatcl.github.io/advds/diagrams/dimred/projection-examples.svg" class="" width="" style="vertical-align:middle;">

Useful projection strategies:

1.  Plot consecutive pairs of dimensions
2.  Project onto principal components
3.  Look at random projections
4.  Examine projections that show unusual patterns
5.  Consider targeted projections based on feature knowledge

## Evaluating Neighborhoods

Understanding local structure is crucial for many methods:

Important neighborhood checks:

1.  Are neighbors sensible for different values of $k$?
2.  Are there isolated points or clusters?
3.  Do distances make sense locally?
4.  Are there boundary effects?
5.  How stable are the neighborhoods?

## Common Pitfalls

Be aware of common issues that can affect results:

Key things to watch out for:

1.  Feature scaling can dramatically affect results
2.  Outliers can dominate the analysis
3.  Choice of distance metric matters
4.  Single visualizations can be misleading
5.  Linear methods might miss important structure

## Validation Strategies

Always validate your dimensionality reduction:

Useful validation approaches:

1.  Compare results from different methods
2.  Check if known relationships are preserved
3.  Verify whether the reduction preserves important structure
4.  Test stability with different subsets of data
5.  Use domain knowledge to validate results

## Implementation Guidelines

When implementing dimensionality reduction:

Follow these steps:

1.  Begin with simple methods like PCA
2.  Scale and preprocess data appropriately
3.  Check intermediate results at each step
4.  Validate assumptions about your data
5.  Document all preprocessing and parameter choices

## Thanks!

For more information on these subjects and more you might want to check
the following resources.

-   company: [Trent AI](https://trent.ai)
-   book: [The Atomic
    Human](https://www.penguin.co.uk/books/455130/the-atomic-human-by-lawrence-neil-d/9780241625248)
-   twitter: [@lawrennd](https://twitter.com/lawrennd)
-   podcast: [The Talking Machines](http://thetalkingmachines.com)
-   newspaper: [Guardian Profile
    Page](http://www.theguardian.com/profile/neil-lawrence)
-   blog:
    [http://inverseprobability.com](http://inverseprobability.com/blog.html)

::: {.cell .markdown}

## References

<!--https://github.com/neelsoumya/visualization_lecture/blob/main/visualization_lecture.pptx

More material is in the repo:

https://github.com/neelsoumya/visualization_lecture/-->

Bishop, C.M., James, G.D., 1993. Analysis of multiphase flows using
dual-energy gamma densitometry and neural networks. Nuclear Instruments
and Methods in Physics Research A327, 580–593.
<https://doi.org/10.1016/0168-9002(93)90728-Z>

Hinton, G.E., Roweis, S.T., n.d. Stochastic neighbor embedding. pp.
857–864.

Hotelling, H., 1933. Analysis of a complex of statistical variables into
principal components. Journal of Educational Psychology 24, 417–441.

Pearson, K., 1901. On lines and planes of closest fit to systems of
points in space. The London, Edinburgh and Dublin Philosophical Magazine
and Journal of Science, Sixth Series 2, 559–572.

Roweis, S.T., n.d. EM algorithms for PCA and SPCA. pp. 626–632.

Spearman, C.E., 1904. "General intelligence," objectively determined and
measured. The American Journal of Psychology 15, 201–292.

Tipping, M.E., Bishop, C.M., 1999b. Mixtures of probabilistic principal
component analysers. Neural Computation 11, 443–482.

Tipping, M.E., Bishop, C.M., 1999a. Probabilistic principal component
analysis. Journal of the Royal Statistical Society, B 6, 611–622.
<https://doi.org/doi:10.1111/1467-9868.00196>

van der Maaten, L.J.P., Hinton, G.E., 2008. Visualizing data using
t-SNE. Journal of Machine Learning Research 9, 2579–2605.